# Lab: GenAI Cortex AI Functions Part 2

📚 In this lab you will learn and practice the following:

❄️ Use the AI_COMPLETE function for text generation

❄️ Control model output using parameters (temperature, top_p, max_tokens)

❄️ Shape LLM responses through effective prompting techniques

❄️ Build model processing pipelines using CTEs

❄️ Use TRY_COMPLETE for error-safe LLM calls

📌 **Note**:

Due to **regional workload spikes**, there may be **latency** with some of the steps. In the real world, for consistent performance, customers can explore [**Provisioned Throughput**](https://docs.snowflake.com/en/user-guide/snowflake-cortex/provisioned-throughput).

If you find your queries are running for more than 5 minutes, cancel and come back and try them later.

---

### 🤖 Use CoCo as you go!

> **💡 TIP 1**: Use CoCo to explain complex SQL statements. Select any query and ask *"Explain this SQL"* to get a plain-language breakdown of what it does.
>
> **💡 TIP 2**: Want to learn more about any function? Ask CoCo *"What does [function name] do?"* to get its syntax, supported options, and examples.
>
> **💡 TIP 3**: If you encounter a deprecated model error, ask CoCo *"Replace deprecated models in this notebook with current similar low-cost alternatives"* and it will fix them for you.

---

## Connect to a Service

Before running cells in this notebook, you must connect to a compute service.

**First time (create a new service):**
1. Click the **Connect** button at the top of this notebook
2. Click **Create Service** — a default name like `{{user}}_SERVICE1` will be suggested
3. Click **Service Settings** and select `ALLOW_ALL_EAI` as the external access integration
4. Leave other settings as default and click **Create**
5. Wait for the service to reach a **READY** state

**Returning (service already exists):**
1. Click the **Connect** button
2. Select your existing service from the list

Once connected, you can run Python and SQL cells interactively.

## Introduction

This lab explores **AI_COMPLETE** and **TRY_COMPLETE**, the flexible, general-purpose LLM functions in Snowflake Cortex that let you choose your model, control output parameters, shape responses through prompting, and build multi-step AI pipelines directly in SQL.

## AI_COMPLETE

Generates a text completion for a given prompt using a supported large language model. Supports model selection, output parameter control, image analysis (covered in Part 3), and multi-model CTE pipelines.

### Getting started with AI_COMPLETE.

Compare responses from two different models - llama3.1-8b and claude-haiku-4-5 - for the same question to see how model choice affects tone and verbosity.

### Set up your current context for the role, database, schema and warehouse.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
your_db = user + '_genai_db'
print('Your current CONTEXT information:')
print(session)

In [ ]:
%%sql -r Set_up_your_current_context_for_the_sql
USE ROLE genai_role;
USE DATABASE {{user}}_genai_db;
USE SCHEMA raw;
USE WAREHOUSE {{user}}_genai_wh;
ALTER SESSION SET query_tag = '{{user}} lab - TOPIC: LLM Functions Part 2';
SHOW PARAMETERS LIKE 'query_tag' in session 
->> SELECT "value" AS query_tag FROM $1;

### Simple prompt interactions.

Run the SQL cell below to compare the responses from the two models side by side.

In [ ]:
%%sql -r Simple_prompt_interactions_sql
SET prompt = 'What is Snowflake?';

SELECT
  'llama3.1-8b' AS model_name,
  AI_COMPLETE('llama3.1-8b', $prompt) AS response
UNION ALL
SELECT
  'claude-haiku-4-5' AS model_name,
  AI_COMPLETE('claude-haiku-4-5', $prompt) AS response;

### Business scenario: draft a response to a negative review.

Set two session variables - **prompt_a** (instructions for the model) and **prompt_b** (the review text) - then concatenate them as the prompt.

In [ ]:
%%sql -r Accessing_model_reasoning_set_prompts_sql
SET prompt_a = 'You are a customer support agent for a travel company. Analyze reviews within <review> tags. If you encounter a negative review, draft an email to the customer apologizing for the issue saying how the business will make it right.';

SET prompt_b = '<review> The ride was incredibly bumpy and uncomfortable. The basket was cramped, with barely enough room to move around. The pilot could not maintain a steady altitude, we felt uneasy and nauseous. During the landing we were jolted around </review>';

In [ ]:
%%sql -r Accessing_model_reasoning_execute_sql
SELECT AI_COMPLETE('claude-haiku-4-5', $prompt_a || $prompt_b ) AS complete_review;

In [ ]:
# Format output from previous cell
df = Accessing_model_reasoning_execute_sql.to_pandas()

# Get the assessment string from the DataFrame
generated_response = df["COMPLETE_REVIEW"].iloc[0]

corrected_text = generated_response.replace('\\n', '\n')

print(corrected_text)

### Shaping output via prompting.

Change the output format by adjusting the instruction in prompt_a - request JSON format, then a CEO bullet-point action list, using the same review text each time.

In [ ]:
%%sql -r Shaping_output_json_sql
SET prompt_a = 'You are a customer support agent for a travel company. Analyze reviews within <review> tags. For negative reviews, draft an email to the customer apologizing for the issue. Provide a summary of the issue in JSON format.';

SELECT AI_COMPLETE('claude-haiku-4-5', $prompt_a || $prompt_b ) AS complete_review;

In [ ]:
# Format output from previous cell
df = Shaping_output_json_sql.to_pandas()

# Get the assessment string from the DataFrame
wine_assessment = df["COMPLETE_REVIEW"].iloc[0]

corrected_text = wine_assessment.replace('\\n', '\n')

print(corrected_text)

In [ ]:
%%sql -r Shaping_output_bullets_sql
SET prompt_a = 'Act as the CEO of the travel company. Analyze reviews. If you encounter a negative review, provide a bulleted list of meeting action items to discuss with the team to address this.';

SELECT AI_COMPLETE('claude-haiku-4-5', $prompt_a || $prompt_b ) AS complete_review;


In [ ]:
# Format output from previous cell
df = Shaping_output_bullets_sql.to_pandas()

# Get the assessment string from the DataFrame
wine_assessment = df["COMPLETE_REVIEW"].iloc[0]

corrected_text = wine_assessment.replace('\\n', '\n')

print(corrected_text)

### Model processing pipeline.

Chain two models in a CTE: llama3.1-8b drafts a customer apology email, then qwen3-32b rewrites it through a legal lens to reduce liability exposure.

In [ ]:
%%sql -r Model_pipeline_set_prompts_sql
SET prompt_a = 'You are a customer support agent for a travel company. Analyze reviews within <review> tags. If you encounter a negative review, draft an email to the customer apologizing for the issue and saying how the business will make it right.';

SET prompt_b = '<review> The ride was incredibly bumpy and uncomfortable. The basket was cramped, with barely enough room to move around. The pilot could not maintain a steady altitude, we felt uneasy and nauseous. During the landing we were jolted around </review>';

In the CTE below, the first model drafts the email and the second reviews and rewrites it. Run it and compare the original draft against the legally-tempered output.

In [ ]:
%%sql -r Model_pipeline_execute_sql
WITH draft AS ( -- in this initial pass we use the llama3.1-8b to draft an email response
    SELECT AI_COMPLETE('llama3.1-8b', $prompt_a || $prompt_b ) as response
)
SELECT -- we then have another model, qwen3-32b, act on that output with a different set of instructions
    AI_COMPLETE('qwen3-32b', 
                              'Act as an executive legal assistant. Review the following email draft. ' ||
                              'Rewrite to ensure the company is not exposed to a legal lawsuit. ' ||
                              '<email> ' ||
                              response ||
                              '</email>' 
                             ) AS complete_review
FROM draft;

In [ ]:

df = Model_pipeline_execute_sql.to_pandas()

# Get the assessment string from the DataFrame
wine_assessment = df["COMPLETE_REVIEW"].iloc[0]

corrected_text = wine_assessment.replace('\\n', '\n')

print(corrected_text)

### Model parameters for AI_COMPLETE.

| Parameter | Default | Effect |
| :--- | :--- | :--- |
| `temperature` | 0 | 0 = deterministic, 1 = diverse/random |
| `top_p` | 0 | Alternative diversity control (use instead of temperature) |
| `max_tokens` | 4096 | Truncates output at this token count |
| `guardrails` | FALSE | Filters harmful or unsafe responses |

### Increasing variability of responses.

Set `temperature: 0.7` and `show_details: true` - the response includes token usage metadata alongside the generated text.

In [ ]:
%%sql -r Increasing_variability_of_responses_sql
SELECT AI_COMPLETE(
    model => 'llama3.1-8b',
    prompt => 'As a travel company that organizes activities for travelers, what role should data play in the operations of my business if I\'m looking to expand my business??',
    model_parameters => {
        'temperature': 0.7
    },
    show_details => true
) AS response;

In [ ]:
# Format output from previous cell
df = Increasing_variability_of_responses_sql.to_pandas()

# Get the assessment string from the DataFrame
generated_response = df["RESPONSE"].iloc[0]

corrected_text = generated_response.replace('\\n', '\n')

print(corrected_text)

The `show_details: true` flag returns the full JSON response including token counts. The next cell extracts just the message text using JSON path notation.

In [ ]:
%%sql -r Variability_extract_message_sql
-- Define the Common Table Expression (CTE) named 'response_cte'
WITH response_cte AS (
  SELECT
    AI_COMPLETE(
      model => 'llama3.1-8b',
      prompt => 'As a travel company that organizes activities for travelers, what role should data play in the operations of my business if I\'m looking to expand my business??',
      model_parameters => { 'temperature': 0.2 },
      show_details => true
    ) AS response
)
-- Now, select from the CTE you just created
SELECT
  response :choices [0].messages :: STRING AS message
FROM
  response_cte;


In [ ]:
# Format output from previous cell
df = Variability_extract_message_sql.to_pandas()

# Get the assessment string from the DataFrame
generated_response = df["MESSAGE"].iloc[0]

corrected_text = generated_response.replace('\\n', '\n')

print(corrected_text)

### Restricting output with max_tokens.

Use `max_tokens` to cap output length - useful for controlling costs on verbose prompts. The first cell truncates at 25 tokens; the second applies no limit across the review table.

In [ ]:
%%sql -r Max_tokens_truncate_sql
SELECT AI_COMPLETE(
    'llama3.1-8b',
    'Draft me an advertising campaign to promote hot air balloon rides. Make it more than 1000 words..',
    {'max_tokens': 25}
) AS truncated_response;

In [ ]:

df = Max_tokens_truncate_sql.to_pandas()

# Get the assessment string from the DataFrame
generated_response = df["TRUNCATED_RESPONSE"].iloc[0]

corrected_text = generated_response.replace('\\n', '\n')

print(corrected_text)

**OPTIONAL** - Apply a health-and-safety analysis prompt across reviews in the traveler_activity table. Each review gets an individual AI assessment. Limited to 5 rows to keep runtime short.

In [ ]:
%%sql -r Max_tokens_health_safety_sql
-- OPTIONAL: This runs AI_COMPLETE on every row and may take several minutes.
-- Remove the LIMIT to process all rows if you have time.
SELECT AI_COMPLETE(
    'llama3.3-70b',
    'You are a senior health and safety assessment officer who provides recommendations to owners of travel companies who organize travel activities according to industry best practices in a warm conversational tone. Be succinct. Response of 150 words max. Analyze the provided traveler reviews outlining any health and safety concerns. If there are no health and safety issues detected say so. <review>' || review_text || '</review>',
    {}
) AS response
FROM {{user}}_genai_db.presentation.traveler_activity
LIMIT 5;

## Helper Function: TRY_COMPLETE

Works identically to AI_COMPLETE but returns **NULL** instead of raising an error on failure - ideal for production pipelines where a single bad row should not halt the entire query.

### Getting started with TRY_COMPLETE.

Run TRY_COMPLETE with a valid model first to confirm it works the same as AI_COMPLETE, then compare error behavior using an invalid model name.

### Simple prompt interaction.

Run TRY_COMPLETE with a valid model - the output should match what AI_COMPLETE would return.

In [ ]:
%%sql -r Simple_prompt_interaction_sql
SELECT SNOWFLAKE.CORTEX.TRY_COMPLETE('llama3.1-8b', [{'role': 'user', 'content': 'What are the governance features of Snowflake? Be succinct. Limit to 500 words'}], {'max_tokens': 600}) AS governance_features;

### Compare AI_COMPLETE vs TRY_COMPLETE error handling.

Pass an invalid model name (`llama3.1-8bS`) to both functions - AI_COMPLETE raises an error, TRY_COMPLETE silently returns NULL.

In [ ]:
%%sql -r TRY_COMPLETE_AI_COMPLETE_error_sql
SELECT AI_COMPLETE(
    'llama3.1-8bS', -- invalid model name. error reported
    'how do you make wine from grapes?',
    {
        'temperature': 0.7,
        'max_tokens': 5
    }
);

In [ ]:
%%sql -r dataframe_1


In [ ]:
%%sql -r TRY_COMPLETE_null_response_sql
SELECT SNOWFLAKE.CORTEX.TRY_COMPLETE(
    'llama3.1-8bS', -- invalid model name. but no error produced
    [
        {
            'role': 'user',
            'content': 'how do you make wine from grapes?.'
        }
    ],
    {
        'temperature': 0.7,
        'max_tokens': 5
    }
);

## 🎯 Challenge Questions

Test your understanding of the concepts covered in this lab.

In [ ]:
from snowflake.snowpark.context import get_active_session
from IPython.display import display, HTML

session = get_active_session()

quiz_data = [
    {"q": "What does the temperature parameter in AI_COMPLETE control?", "options": ["A) The speed of response generation", "B) The number of tokens in the output", "C) The temperature parameter controls the randomness and creativity of responses", "D) The language of the response"], "hash": "3cd3f8730d514fcf60363f308babb15c"},
    {"q": "What is the purpose of the max_tokens parameter in AI_COMPLETE?", "options": ["A) It sets the minimum response length", "B) MAX_TOKENS limits the length of the generated response", "C) It controls the number of input tokens allowed", "D) It determines how many models to query"], "hash": "b31bd28925b5319a2ca8bb9033cae0a4"},
    {"q": "Which feature does AI_COMPLETE support for building conversational applications?", "options": ["A) It automatically stores conversation history in a table", "B) It requires a separate memory service", "C) It only supports single-turn interactions", "D) AI_COMPLETE supports stateful conversations using message history"], "hash": "ff9519732382733c6ef3af43f0434f24"},
    {"q": "What is the key difference between AI_COMPLETE and TRY_COMPLETE?", "options": ["A) TRY_COMPLETE returns NULL instead of throwing an error on failure", "B) TRY_COMPLETE is faster than AI_COMPLETE", "C) TRY_COMPLETE supports more LLM models", "D) TRY_COMPLETE automatically retries failed requests"], "hash": "6501c63c2fe353b1577e31231684c145"},
    {"q": "Why might the same prompt produce different results when using different LLM models?", "options": ["A) Because of network latency differences", "B) Because Snowflake randomizes responses", "C) Different LLM models can produce varying responses for the same prompt", "D) Because of caching mechanisms"], "hash": "e1e247b5337116b92804f5909751cd11"},
]

results_map = {}
for qi, item in enumerate(quiz_data):
    results_map[qi] = {}
    for opt in item["options"]:
        letter = opt[0]
        escaped_opt = opt.replace("'", "''")
        result = session.sql(f"CALL genai_db.resources.quiz_temp('{item['hash']}', '{escaped_opt}', 'False')").collect()
        feedback = result[0][0]
        is_correct = 'Correct' in feedback or '✅' in feedback
        results_map[qi][letter] = (feedback, is_correct)

html = """<style>
.cq { margin: 20px 0; padding: 16px; border: 1px solid #d0d0d0; border-radius: 10px; background: #fafafa; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif; font-size: 14px; }
.cq h4 { font-family: inherit; }
.cq input[type="radio"] { display: none; }
.cq .lbl { display: block; padding: 8px 12px; border-radius: 6px; cursor: pointer; font-family: inherit; }
.cq .lbl:hover { background: #e8f0fe; }
.cq input[type="radio"]:checked + .lbl { border-color: #1a73e8; background: #e8f0fe; font-weight: 600; }
.cq .fb { display: none; padding: 6px 12px; margin-top: 2px; border-radius: 4px; font-weight: 600; font-family: inherit; }
.cq input[type="radio"]:checked + .lbl + .fb { display: block; }
.cq .fb.ok { background: #e6f4ea; color: #1e7e34; }
.cq .fb.no { background: #fce8e6; color: #c62828; }
</style>"""

for qi, item in enumerate(quiz_data):
    html += f'<div class="cq"><h4>Q{qi+1}: {item["q"]}</h4>'
    for opt in item["options"]:
        letter = opt[0]
        feedback, is_correct = results_map[qi][letter]
        css_class = 'ok' if is_correct else 'no'
        uid = f'cq{qi}_{letter}'
        html += f'<div class="opt"><input type="radio" name="cq{qi}" id="{uid}">'
        html += f'<label class="lbl" for="{uid}">{opt}</label>'
        html += f'<div class="fb {css_class}">{letter}) {feedback}</div></div>'
    html += '</div>'

display(HTML(html))


## Key Takeaways

❄️ **AI_COMPLETE** is a general-purpose LLM function that supports model selection, output parameter control, and multi-step pipelines. It is powerful on its own and even more so when combined with CTEs.

❄️ **Model parameters** (temperature, top_p, max_tokens, guardrails) give you precise control over output variability, length, and safety. Experiment to find the right settings for each use case.

❄️ **Prompt engineering** directly impacts response quality. Structure your prompts clearly by specifying the role, context, and desired output format to get consistent, usable results.

❄️ **CTE-based model pipelines** let you chain multiple AI_COMPLETE calls in a single SQL query, passing outputs from one model as inputs to another for multi-step processing.

❄️ **TRY_COMPLETE** returns NULL instead of raising an error, making it safe to use in batch queries where individual failures should not halt the entire pipeline.